# Chapter 12.4: Serving Reasoning Models -- Chain-of-Thought, R1-style

## Overview

Reasoning models (DeepSeek-R1, OpenAI o1/o3) generate long chains-of-thought before producing final answers. This creates unique serving challenges:

- **Unpredictable output length**: 100 tokens to 100K+ tokens
- **High variance**: Some queries trigger extensive reasoning, others don't
- **Thinking tokens vs response tokens**: Users pay for thinking but may want to control budgets
- **Long decode times**: Decode phase dominates latency

### Learning Objectives
1. Understand reasoning model workload characteristics
2. Simulate scheduling challenges for variable-length outputs
3. Implement thinking token budget control
4. Design specialized schedulers for reasoning workloads

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part12/chapter_12.4_reasoning_models.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part12/chapter_12.4_reasoning_models.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import deque
import heapq
import time

np.random.seed(42)
plt.style.use('default')
print("Reasoning model serving environment ready.")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import numpy as np

def draw_reasoning_model_output_pattern():
    """Reasoning model output pattern: <thinking>long</thinking><answer>short</answer> with token count."""
    fig, ax = plt.subplots(figsize=(16, 7))
    ax.set_xlim(0, 18); ax.set_ylim(0, 8); ax.axis('off')
    ax.set_title('Reasoning Model Output Pattern: Thinking vs Answer Tokens',
                 fontsize=15, fontweight='bold', pad=15)

    # Input prompt
    prompt = FancyBboxPatch((0.5, 5.5), 3.0, 1.5, boxstyle='round,pad=0.12',
                            facecolor='#4A90D9', edgecolor='white', linewidth=2, alpha=0.85)
    ax.add_patch(prompt)
    ax.text(2.0, 6.25, 'User Prompt\n(100-2000 tok)', ha='center', va='center',
            fontsize=9, fontweight='bold', color='white')

    ax.annotate('', xy=(3.8, 6.25), xytext=(3.5, 6.25),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))

    # Thinking block (large)
    thinking = FancyBboxPatch((4.0, 4.5), 8.0, 3.0, boxstyle='round,pad=0.15',
                              facecolor='#F39C12', edgecolor='white', linewidth=2, alpha=0.85)
    ax.add_patch(thinking)
    ax.text(8.0, 6.5, '<thinking>', ha='center', va='center',
            fontsize=12, fontweight='bold', color='white', family='monospace')
    ax.text(8.0, 5.5, 'Long internal reasoning chain...\nStep 1: Analyze the problem\nStep 2: Consider approaches\nStep 3: Work through solution\n...',
            ha='center', va='center', fontsize=8, color='white', alpha=0.9)
    ax.text(8.0, 4.7, '</thinking>', ha='center', va='center',
            fontsize=10, fontweight='bold', color='white', family='monospace')

    ax.annotate('', xy=(12.3, 6.25), xytext=(12.0, 6.25),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))

    # Answer block (small)
    answer = FancyBboxPatch((12.5, 5.0), 3.5, 2.0, boxstyle='round,pad=0.12',
                            facecolor='#27AE60', edgecolor='white', linewidth=2, alpha=0.85)
    ax.add_patch(answer)
    ax.text(14.25, 6.3, '<answer>', ha='center', va='center',
            fontsize=10, fontweight='bold', color='white', family='monospace')
    ax.text(14.25, 5.7, 'Short final\nanswer', ha='center', va='center',
            fontsize=9, color='white')
    ax.text(14.25, 5.15, '</answer>', ha='center', va='center',
            fontsize=10, fontweight='bold', color='white', family='monospace')

    # Token count comparison bar chart at bottom
    categories = ['Simple QA', 'Math Problem', 'Complex Reasoning', 'Code Generation']
    thinking_tokens = [200, 2000, 10000, 5000]
    answer_tokens = [50, 200, 500, 1000]

    bar_y = 1.5
    bar_height = 0.5
    max_total = max(t + a for t, a in zip(thinking_tokens, answer_tokens))

    ax.text(9, 3.5, 'Token Count by Task Type', ha='center', fontsize=11,
            fontweight='bold', color='#333')

    for i, (cat, think, ans) in enumerate(zip(categories, thinking_tokens, answer_tokens)):
        y = bar_y + (3 - i) * 0.7
        think_w = 10 * think / max_total
        ans_w = 10 * ans / max_total

        # Thinking bar
        rect_t = FancyBboxPatch((3, y), think_w, bar_height, boxstyle='round,pad=0.02',
                                facecolor='#F39C12', edgecolor='none', alpha=0.7)
        ax.add_patch(rect_t)
        # Answer bar
        rect_a = FancyBboxPatch((3 + think_w, y), ans_w, bar_height, boxstyle='round,pad=0.02',
                                facecolor='#27AE60', edgecolor='none', alpha=0.7)
        ax.add_patch(rect_a)

        ax.text(2.9, y + bar_height/2, cat, ha='right', va='center', fontsize=8, color='#333')
        ax.text(3 + think_w + ans_w + 0.2, y + bar_height/2,
                f'{think}+{ans} = {think+ans} tok', ha='left', va='center', fontsize=7, color='#555')

    # Legend
    ax.text(14, 2.5, 'Thinking tokens', fontsize=8, color='#F39C12', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#FFF3E0', edgecolor='#F39C12'))
    ax.text(14, 1.8, 'Answer tokens', fontsize=8, color='#27AE60', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#E8F5E9', edgecolor='#27AE60'))

    plt.tight_layout()
    plt.savefig('reasoning_output_pattern.png', dpi=150, bbox_inches='tight')
    plt.show()

draw_reasoning_model_output_pattern()

## 1. Reasoning Model Workload Characteristics

In [ ]:
class ReasoningWorkloadGenerator:
    """Generate realistic reasoning model workloads."""

    # Task categories with different reasoning patterns
    TASK_PROFILES = {
        'simple_qa': {
            'prompt_len': (100, 500),
            'thinking_tokens': (50, 500),     # Short reasoning
            'response_tokens': (20, 200),
            'probability': 0.3,
        },
        'math_problem': {
            'prompt_len': (200, 1000),
            'thinking_tokens': (500, 5000),   # Moderate reasoning
            'response_tokens': (50, 500),
            'probability': 0.25,
        },
        'complex_reasoning': {
            'prompt_len': (500, 2000),
            'thinking_tokens': (2000, 20000),  # Long reasoning chains
            'response_tokens': (100, 2000),
            'probability': 0.2,
        },
        'code_generation': {
            'prompt_len': (300, 2000),
            'thinking_tokens': (1000, 10000),
            'response_tokens': (200, 5000),
            'probability': 0.15,
        },
        'creative_writing': {
            'prompt_len': (100, 500),
            'thinking_tokens': (200, 2000),
            'response_tokens': (500, 10000),
            'probability': 0.1,
        },
    }

    def generate(self, num_requests: int, rate: float) -> List[dict]:
        """Generate workload with Poisson arrivals."""
        tasks = list(self.TASK_PROFILES.keys())
        probs = [self.TASK_PROFILES[t]['probability'] for t in tasks]

        requests = []
        current_time = 0.0

        for i in range(num_requests):
            task = np.random.choice(tasks, p=probs)
            profile = self.TASK_PROFILES[task]

            prompt_len = np.random.randint(*profile['prompt_len'])
            # Use log-normal for thinking tokens (heavy tail)
            thinking_mean = np.mean(profile['thinking_tokens'])
            thinking_std = (profile['thinking_tokens'][1] - profile['thinking_tokens'][0]) / 4
            thinking_tokens = int(np.clip(
                np.random.lognormal(np.log(thinking_mean), 0.8),
                profile['thinking_tokens'][0],
                profile['thinking_tokens'][1] * 2  # Allow some outliers
            ))
            response_tokens = np.random.randint(*profile['response_tokens'])

            inter_arrival = np.random.exponential(1.0 / rate)
            current_time += inter_arrival

            requests.append({
                'id': i,
                'task': task,
                'arrival_time': current_time,
                'prompt_len': prompt_len,
                'thinking_tokens': thinking_tokens,
                'response_tokens': response_tokens,
                'total_output': thinking_tokens + response_tokens,
            })

        return requests


gen = ReasoningWorkloadGenerator()
workload = gen.generate(1000, rate=5)

# Analyze workload statistics
print(f"Reasoning Workload Statistics ({len(workload)} requests):")
print(f"{'Metric':>25} {'Mean':>10} {'Median':>10} {'P95':>10} {'P99':>10} {'Max':>10}")
print("-" * 80)
for metric in ['prompt_len', 'thinking_tokens', 'response_tokens', 'total_output']:
    vals = [r[metric] for r in workload]
    print(f"{metric:>25} {np.mean(vals):>10.0f} {np.median(vals):>10.0f} "
          f"{np.percentile(vals, 95):>10.0f} {np.percentile(vals, 99):>10.0f} {np.max(vals):>10.0f}")

print(f"\nTask distribution:")
from collections import Counter
task_counts = Counter(r['task'] for r in workload)
for task, count in sorted(task_counts.items(), key=lambda x: -x[1]):
    avg_think = np.mean([r['thinking_tokens'] for r in workload if r['task'] == task])
    print(f"  {task:>25}: {count:>4} requests, avg thinking={avg_think:.0f} tokens")

In [ ]:
# Visualize workload distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Output length distribution
total_outputs = [r['total_output'] for r in workload]
axes[0, 0].hist(total_outputs, bins=100, density=True, alpha=0.7, color='steelblue')
axes[0, 0].axvline(np.mean(total_outputs), color='red', linestyle='--', label=f'Mean={np.mean(total_outputs):.0f}')
axes[0, 0].axvline(np.percentile(total_outputs, 95), color='orange', linestyle='--', label=f'P95={np.percentile(total_outputs, 95):.0f}')
axes[0, 0].set_xlabel('Total Output Tokens')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Output Length Distribution (Heavy-tailed)')
axes[0, 0].legend()

# Thinking vs Response tokens
thinking = [r['thinking_tokens'] for r in workload]
response = [r['response_tokens'] for r in workload]
axes[0, 1].scatter(thinking, response, alpha=0.3, s=10,
                    c=[list(ReasoningWorkloadGenerator.TASK_PROFILES.keys()).index(r['task'])
                       for r in workload], cmap='tab10')
axes[0, 1].set_xlabel('Thinking Tokens')
axes[0, 1].set_ylabel('Response Tokens')
axes[0, 1].set_title('Thinking vs Response Tokens')

# Per-task output distribution
tasks = list(ReasoningWorkloadGenerator.TASK_PROFILES.keys())
for task in tasks:
    task_outputs = [r['total_output'] for r in workload if r['task'] == task]
    axes[1, 0].hist(task_outputs, bins=50, alpha=0.5, density=True, label=task)
axes[1, 0].set_xlabel('Total Output Tokens')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Output Distribution by Task')
axes[1, 0].legend(fontsize=7)

# Thinking/Total ratio
ratios = [r['thinking_tokens'] / r['total_output'] for r in workload]
axes[1, 1].hist(ratios, bins=50, density=True, alpha=0.7, color='green')
axes[1, 1].set_xlabel('Thinking / Total Ratio')
axes[1, 1].set_ylabel('Density')
axes[1, 1].set_title(f'Thinking Token Ratio (Mean={np.mean(ratios):.2f})')

plt.tight_layout()
plt.savefig('reasoning_workload.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Demo: Simulate Reasoning Model Serving

In [ ]:
class ReasoningModelServer:
    """Simulate serving a reasoning model with different scheduling strategies."""

    def __init__(self, num_gpus: int = 4, max_batch_size: int = 32,
                 decode_tpot_ms: float = 30.0,  # Time per output token per request
                 prefill_speed_tokens_per_s: float = 50000):
        self.num_gpus = num_gpus
        self.max_batch = max_batch_size
        self.decode_tpot = decode_tpot_ms / 1000  # Convert to seconds
        self.prefill_speed = prefill_speed_tokens_per_s

    def serve_fcfs(self, requests: List[dict]) -> List[dict]:
        """First-Come-First-Serve scheduling."""
        import copy
        results = [copy.deepcopy(r) for r in requests]

        gpu_available = [0.0] * self.num_gpus

        for req in results:
            gpu_idx = int(np.argmin(gpu_available))
            start = max(req['arrival_time'], gpu_available[gpu_idx])

            # Prefill
            prefill_time = req['prompt_len'] / self.prefill_speed
            req['prefill_end'] = start + prefill_time

            # Decode (thinking + response)
            decode_time = req['total_output'] * self.decode_tpot
            req['complete_time'] = req['prefill_end'] + decode_time
            req['ttft'] = req['prefill_end'] - req['arrival_time']
            req['total_latency'] = req['complete_time'] - req['arrival_time']

            gpu_available[gpu_idx] = req['complete_time']

        return results

    def serve_sjf(self, requests: List[dict]) -> List[dict]:
        """Shortest-Job-First: prioritize requests with shorter expected output.
        NOTE: This requires output length prediction."""
        import copy
        sorted_reqs = sorted(requests, key=lambda r: r['total_output'])
        return self.serve_fcfs(sorted_reqs)

    def serve_preemptive(self, requests: List[dict],
                          preemption_threshold: int = 5000) -> List[dict]:
        """Preemptive scheduling: pause long-running requests to admit new ones."""
        import copy
        results = [copy.deepcopy(r) for r in requests]

        gpu_available = [0.0] * self.num_gpus
        # Track active requests per GPU
        active = [None] * self.num_gpus

        pending = deque(sorted(results, key=lambda r: r['arrival_time']))
        completed = []

        current_time = 0.0
        while pending or any(a is not None for a in active):
            # Find next event
            next_arrival = pending[0]['arrival_time'] if pending else float('inf')
            next_completion = min(
                (gpu_available[i] for i in range(self.num_gpus) if active[i] is not None),
                default=float('inf')
            )
            current_time = min(next_arrival, next_completion)

            if current_time == float('inf'):
                break

            # Complete finished requests
            for i in range(self.num_gpus):
                if active[i] is not None and gpu_available[i] <= current_time:
                    active[i]['complete_time'] = gpu_available[i]
                    active[i]['total_latency'] = active[i]['complete_time'] - active[i]['arrival_time']
                    completed.append(active[i])
                    active[i] = None

            # Assign pending requests to free GPUs
            while pending and pending[0]['arrival_time'] <= current_time:
                req = pending.popleft()
                free_gpus = [i for i in range(self.num_gpus) if active[i] is None]
                if free_gpus:
                    gpu = free_gpus[0]
                    prefill_time = req['prompt_len'] / self.prefill_speed
                    req['prefill_end'] = current_time + prefill_time
                    req['ttft'] = req['prefill_end'] - req['arrival_time']
                    decode_time = req['total_output'] * self.decode_tpot
                    gpu_available[gpu] = req['prefill_end'] + decode_time
                    active[gpu] = req
                else:
                    pending.appendleft(req)
                    break

            # Safety: advance time if stuck
            if all(a is not None for a in active) and pending:
                next_free = min(gpu_available)
                if next_free > current_time:
                    current_time = next_free
                    continue

            if not pending and all(a is None for a in active):
                break

            # Avoid infinite loops
            if current_time > 1e6:
                break

        # Collect remaining active
        for i in range(self.num_gpus):
            if active[i] is not None:
                active[i]['complete_time'] = gpu_available[i]
                active[i]['total_latency'] = active[i]['complete_time'] - active[i]['arrival_time']
                completed.append(active[i])

        return completed


# Run simulations
server = ReasoningModelServer(num_gpus=4, max_batch_size=32, decode_tpot_ms=30)
small_workload = workload[:200]

fcfs_results = server.serve_fcfs(small_workload)
# For SJF, we simulate having perfect output length prediction
sjf_server = ReasoningModelServer(num_gpus=4, max_batch_size=32, decode_tpot_ms=30)
sjf_results = sjf_server.serve_sjf(small_workload)

def summarize_results(results, name):
    latencies = [r['total_latency'] for r in results if 'total_latency' in r]
    ttfts = [r['ttft'] for r in results if 'ttft' in r]
    if not latencies:
        return
    print(f"\n{name}:")
    print(f"  Avg latency: {np.mean(latencies):.2f}s")
    print(f"  P50 latency: {np.median(latencies):.2f}s")
    print(f"  P99 latency: {np.percentile(latencies, 99):.2f}s")
    print(f"  Avg TTFT: {np.mean(ttfts)*1000:.1f}ms")
    throughput = len(results) / (max(r['complete_time'] for r in results if 'complete_time' in r) -
                                  min(r['arrival_time'] for r in results))
    print(f"  Throughput: {throughput:.2f} req/s")
    return {'avg_lat': np.mean(latencies), 'p99_lat': np.percentile(latencies, 99),
            'throughput': throughput}

print("Scheduling Strategy Comparison:")
fcfs_metrics = summarize_results(fcfs_results, "FCFS")
sjf_metrics = summarize_results(sjf_results, "SJF (perfect prediction)")

In [ ]:
# Visualize scheduling impact
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Latency distribution comparison
for results, name, color in [(fcfs_results, 'FCFS', 'blue'),
                               (sjf_results, 'SJF', 'green')]:
    lats = [r['total_latency'] for r in results if 'total_latency' in r]
    axes[0].hist(lats, bins=50, alpha=0.5, density=True, color=color, label=name)

axes[0].set_xlabel('Total Latency (s)')
axes[0].set_ylabel('Density')
axes[0].set_title('Latency Distribution')
axes[0].legend()

# Latency vs output length
for results, name, color in [(fcfs_results, 'FCFS', 'blue'),
                               (sjf_results, 'SJF', 'green')]:
    outputs = [r['total_output'] for r in results if 'total_latency' in r]
    lats = [r['total_latency'] for r in results if 'total_latency' in r]
    axes[1].scatter(outputs, lats, alpha=0.3, s=10, color=color, label=name)

axes[1].set_xlabel('Total Output Tokens')
axes[1].set_ylabel('Latency (s)')
axes[1].set_title('Latency vs Output Length')
axes[1].legend()

# CDF comparison
for results, name, color in [(fcfs_results, 'FCFS', 'blue'),
                               (sjf_results, 'SJF', 'green')]:
    lats = sorted([r['total_latency'] for r in results if 'total_latency' in r])
    cdf = np.arange(1, len(lats) + 1) / len(lats)
    axes[2].plot(lats, cdf, color=color, linewidth=2, label=name)

axes[2].set_xlabel('Latency (s)')
axes[2].set_ylabel('CDF')
axes[2].set_title('Latency CDF')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reasoning_scheduling.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Demo: Output Length Distribution Analysis

In [ ]:
class OutputLengthPredictor:
    """Predict output length from prompt features for scheduling."""

    def __init__(self):
        self.task_stats = {}  # task -> (mean, std) of output length

    def fit(self, requests: List[dict]):
        """Learn output length distribution per task."""
        from collections import defaultdict
        task_outputs = defaultdict(list)
        for r in requests:
            task_outputs[r['task']].append(r['total_output'])

        for task, outputs in task_outputs.items():
            self.task_stats[task] = {
                'mean': np.mean(outputs),
                'std': np.std(outputs),
                'median': np.median(outputs),
                'p75': np.percentile(outputs, 75),
                'p95': np.percentile(outputs, 95),
            }

    def predict(self, task: str, percentile: float = 50) -> int:
        """Predict output length at given percentile."""
        if task not in self.task_stats:
            return 2000  # Default
        stats = self.task_stats[task]
        if percentile == 50:
            return int(stats['median'])
        elif percentile == 75:
            return int(stats['p75'])
        elif percentile == 95:
            return int(stats['p95'])
        else:
            # Approximate using normal distribution
            from scipy.stats import norm
            z = norm.ppf(percentile / 100)
            return int(stats['mean'] + z * stats['std'])

    def prediction_error(self, requests: List[dict], percentile: float = 50) -> dict:
        """Evaluate prediction quality."""
        errors = []
        over_predictions = 0
        under_predictions = 0

        for r in requests:
            predicted = self.predict(r['task'], percentile)
            actual = r['total_output']
            error = predicted - actual
            errors.append(abs(error) / actual)  # Relative error
            if predicted > actual:
                over_predictions += 1
            else:
                under_predictions += 1

        return {
            'mean_relative_error': np.mean(errors),
            'median_relative_error': np.median(errors),
            'over_prediction_rate': over_predictions / len(requests),
            'under_prediction_rate': under_predictions / len(requests),
        }


# Train predictor on first 500 requests, test on rest
train_set = workload[:500]
test_set = workload[500:]

predictor = OutputLengthPredictor()
predictor.fit(train_set)

print(f"Output Length Statistics per Task:")
print(f"{'Task':>25} {'Mean':>8} {'Median':>8} {'P75':>8} {'P95':>8}")
print("-" * 60)
for task, stats in predictor.task_stats.items():
    print(f"{task:>25} {stats['mean']:>8.0f} {stats['median']:>8.0f} "
          f"{stats['p75']:>8.0f} {stats['p95']:>8.0f}")

print(f"\nPrediction Quality (test set, {len(test_set)} requests):")
for pct in [50, 75, 95]:
    errors = predictor.prediction_error(test_set, pct)
    print(f"  Percentile {pct}: rel_error={errors['mean_relative_error']:.2f}, "
          f"over={errors['over_prediction_rate']:.1%}, under={errors['under_prediction_rate']:.1%}")

## 4. Demo: Thinking Token Budget Control

Control how many tokens the model spends on "thinking" to manage cost and latency.

In [ ]:
class ThinkingBudgetController:
    """Control thinking token budget for reasoning models."""

    def __init__(self, default_budget: int = 5000):
        self.default_budget = default_budget

    def compute_budget(self, task: str, quality_target: str = 'balanced',
                        latency_budget_s: Optional[float] = None,
                        cost_budget_tokens: Optional[int] = None) -> dict:
        """
        Compute thinking budget based on constraints.

        quality_target: 'fast' (minimize thinking), 'balanced', 'thorough'
        """
        quality_multipliers = {
            'fast': 0.3,
            'balanced': 1.0,
            'thorough': 2.0,
        }

        task_base_budgets = {
            'simple_qa': 200,
            'math_problem': 2000,
            'complex_reasoning': 8000,
            'code_generation': 4000,
            'creative_writing': 1000,
        }

        base = task_base_budgets.get(task, self.default_budget)
        budget = int(base * quality_multipliers.get(quality_target, 1.0))

        # Apply latency constraint
        if latency_budget_s is not None:
            decode_tpot = 0.03  # 30ms per token
            max_tokens = int(latency_budget_s / decode_tpot)
            response_estimate = task_base_budgets.get(task, 500) // 4
            budget = min(budget, max_tokens - response_estimate)

        # Apply cost constraint
        if cost_budget_tokens is not None:
            response_estimate = task_base_budgets.get(task, 500) // 4
            budget = min(budget, cost_budget_tokens - response_estimate)

        budget = max(budget, 50)  # Minimum thinking

        return {
            'thinking_budget': budget,
            'estimated_response': task_base_budgets.get(task, 500) // 4,
            'total_budget': budget + task_base_budgets.get(task, 500) // 4,
            'estimated_latency_s': (budget + task_base_budgets.get(task, 500) // 4) * 0.03,
            'estimated_cost_factor': budget / base if base > 0 else 1,
        }


controller = ThinkingBudgetController()

print(f"Thinking Budget Analysis:")
print(f"{'Task':>25} {'Quality':>10} {'Budget':>8} {'Latency':>10} {'Cost Factor':>12}")
print("-" * 70)
for task in ['simple_qa', 'math_problem', 'complex_reasoning']:
    for quality in ['fast', 'balanced', 'thorough']:
        result = controller.compute_budget(task, quality)
        print(f"{task:>25} {quality:>10} {result['thinking_budget']:>8} "
              f"{result['estimated_latency_s']:>8.1f}s {result['estimated_cost_factor']:>10.2f}x")

# Simulate quality vs budget trade-off
def simulate_quality(original_thinking: int, budget: int) -> float:
    """Simulate quality score based on thinking budget.
    More thinking generally helps but with diminishing returns."""
    if budget >= original_thinking:
        return 1.0  # Full quality
    ratio = budget / original_thinking
    # Logarithmic quality curve: rapid improvement initially, then diminishing
    return min(1.0, 0.3 + 0.7 * np.log(1 + ratio * 5) / np.log(6))

budgets = np.arange(50, 10001, 50)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for orig in [500, 2000, 5000, 10000]:
    qualities = [simulate_quality(orig, b) for b in budgets]
    ax1.plot(budgets, qualities, label=f'Original={orig}')

ax1.set_xlabel('Thinking Budget (tokens)')
ax1.set_ylabel('Quality Score')
ax1.set_title('Quality vs Thinking Budget')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Cost-quality Pareto frontier
for task_name, base_thinking in [('math', 2000), ('reasoning', 8000)]:
    costs = budgets * 0.03  # Latency as cost proxy
    qualities = [simulate_quality(base_thinking, b) for b in budgets]
    ax2.plot(costs, qualities, label=task_name)

ax2.set_xlabel('Latency Cost (seconds)')
ax2.set_ylabel('Quality Score')
ax2.set_title('Quality-Latency Trade-off')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('thinking_budget.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Demo: Compare Scheduling Strategies for Reasoning Models

In [ ]:
class MultilevelFeedbackScheduler:
    """MLFQ-inspired scheduler for reasoning models.
    Requests start at high priority and are demoted as they generate more tokens."""

    def __init__(self, num_gpus: int = 4, num_levels: int = 4,
                 level_thresholds: List[int] = [500, 2000, 8000, 32000],
                 decode_tpot_ms: float = 30.0,
                 prefill_speed: float = 50000):
        self.num_gpus = num_gpus
        self.num_levels = num_levels
        self.thresholds = level_thresholds
        self.decode_tpot = decode_tpot_ms / 1000
        self.prefill_speed = prefill_speed

    def serve(self, requests: List[dict]) -> List[dict]:
        """Simulate MLFQ scheduling."""
        import copy
        results = [copy.deepcopy(r) for r in requests]

        # Simple simulation: assign based on expected output length
        gpu_available = [0.0] * self.num_gpus

        # Sort by arrival, then process with priority based on level
        results.sort(key=lambda r: r['arrival_time'])

        for req in results:
            # Determine level based on expected output
            level = 0
            for i, thresh in enumerate(self.thresholds):
                if req['total_output'] > thresh:
                    level = i + 1

            # Higher level = lower priority = wait longer
            # Simple: add priority penalty
            priority_penalty = level * 0.5  # seconds per level

            gpu_idx = int(np.argmin(gpu_available))
            start = max(req['arrival_time'] + priority_penalty, gpu_available[gpu_idx])

            prefill_time = req['prompt_len'] / self.prefill_speed
            req['prefill_end'] = start + prefill_time
            req['ttft'] = req['prefill_end'] - req['arrival_time']

            decode_time = req['total_output'] * self.decode_tpot
            req['complete_time'] = req['prefill_end'] + decode_time
            req['total_latency'] = req['complete_time'] - req['arrival_time']
            req['level'] = level

            gpu_available[gpu_idx] = req['complete_time']

        return results


# Compare all strategies
mlfq = MultilevelFeedbackScheduler(num_gpus=4)
mlfq_results = mlfq.serve(small_workload)

all_strategies = {
    'FCFS': fcfs_results,
    'SJF': sjf_results,
    'MLFQ': mlfq_results,
}

print(f"\nComprehensive Scheduling Comparison:")
print(f"{'Strategy':>10} {'Avg Lat':>10} {'P50 Lat':>10} {'P95 Lat':>10} "
      f"{'P99 Lat':>10} {'Avg TTFT':>10}")
print("-" * 65)
for name, results in all_strategies.items():
    lats = [r['total_latency'] for r in results if 'total_latency' in r]
    ttfts = [r['ttft'] for r in results if 'ttft' in r]
    if lats:
        print(f"{name:>10} {np.mean(lats):>8.1f}s {np.median(lats):>8.1f}s "
              f"{np.percentile(lats, 95):>8.1f}s {np.percentile(lats, 99):>8.1f}s "
              f"{np.mean(ttfts)*1000:>8.0f}ms")

In [ ]:
# Visualize per-task performance
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Overall CDF
ax = axes[0, 0]
for name, results in all_strategies.items():
    lats = sorted([r['total_latency'] for r in results if 'total_latency' in r])
    cdf = np.arange(1, len(lats) + 1) / len(lats)
    ax.plot(lats, cdf, linewidth=2, label=name)
ax.set_xlabel('Latency (s)')
ax.set_ylabel('CDF')
ax.set_title('Overall Latency CDF')
ax.legend()
ax.grid(True, alpha=0.3)

# Per-task median latency
ax = axes[0, 1]
tasks = list(ReasoningWorkloadGenerator.TASK_PROFILES.keys())
x = np.arange(len(tasks))
width = 0.25
for i, (name, results) in enumerate(all_strategies.items()):
    medians = []
    for task in tasks:
        task_lats = [r['total_latency'] for r in results
                     if 'total_latency' in r and r['task'] == task]
        medians.append(np.median(task_lats) if task_lats else 0)
    ax.bar(x + i * width, medians, width, label=name)

ax.set_xticks(x + width)
ax.set_xticklabels(tasks, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Median Latency (s)')
ax.set_title('Median Latency by Task Type')
ax.legend(fontsize=8)

# Thinking vs waiting time
ax = axes[1, 0]
for name, results in [('FCFS', fcfs_results)]:
    valid = [r for r in results if 'total_latency' in r]
    decode_times = [r['total_output'] * 0.03 for r in valid]
    wait_times = [r['total_latency'] - r['total_output'] * 0.03 - r['prompt_len']/50000
                  for r in valid]
    ax.scatter(decode_times, wait_times, alpha=0.3, s=10)
ax.set_xlabel('Decode Time (s)')
ax.set_ylabel('Queue Wait Time (s)')
ax.set_title('Decode vs Queue Wait (FCFS)')
ax.grid(True, alpha=0.3)

# GPU utilization timeline
ax = axes[1, 1]
for name, results in [('FCFS', fcfs_results)]:
    valid = sorted([r for r in results if 'complete_time' in r],
                   key=lambda r: r['arrival_time'])
    times = np.linspace(0, valid[-1]['complete_time'], 200)
    active = [sum(1 for r in valid
                  if r.get('prefill_end', 0) <= t <= r.get('complete_time', 0))
              for t in times]
    ax.plot(times, active, linewidth=1)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Active Requests')
ax.set_title('Active Requests Over Time')
ax.axhline(4, color='red', linestyle='--', alpha=0.5, label='4 GPUs')
ax.legend()

plt.tight_layout()
plt.savefig('scheduling_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Assignment 1: Design a Scheduler for Reasoning Models

Build a scheduler that handles the unique challenges of reasoning workloads.

In [ ]:
# ASSIGNMENT 1: Reasoning Model Scheduler

class ReasoningScheduler:
    """
    TODO: Implement a scheduler optimized for reasoning models.

    Features to implement:
    1. Output length prediction for scheduling decisions
    2. Adaptive batch sizing based on expected decode length
    3. Preemption for excessively long generations
    4. Fair scheduling across different task types
    """

    def __init__(self, num_gpus: int = 4, decode_tpot_ms: float = 30.0,
                 prefill_speed: float = 50000,
                 max_thinking_tokens: int = 20000):
        self.num_gpus = num_gpus
        self.decode_tpot = decode_tpot_ms / 1000
        self.prefill_speed = prefill_speed
        self.max_thinking = max_thinking_tokens
        self.predictor = OutputLengthPredictor()

    def train_predictor(self, historical_data: List[dict]):
        """Train the output length predictor on historical data."""
        self.predictor.fit(historical_data)

    def compute_priority(self, request: dict, current_time: float) -> float:
        """
        TODO: Compute scheduling priority for a request.

        Consider:
        - Predicted output length (shorter = higher priority)
        - Wait time (longer wait = higher priority)
        - Task fairness
        """
        # YOUR CODE HERE
        predicted_len = self.predictor.predict(request['task'], 50)
        wait_time = current_time - request['arrival_time']
        # Aging: increase priority with wait time
        priority = -predicted_len + wait_time * 1000
        return priority

    def should_preempt(self, request: dict, tokens_generated: int) -> bool:
        """
        TODO: Decide whether to preempt (stop) a running request.

        Consider:
        - Has it exceeded thinking budget?
        - Is there a higher priority request waiting?
        """
        # YOUR CODE HERE
        return tokens_generated > self.max_thinking

    def schedule(self, requests: List[dict]) -> List[dict]:
        """
        TODO: Full scheduling implementation.
        """
        import copy
        results = [copy.deepcopy(r) for r in sorted(requests, key=lambda r: r['arrival_time'])]
        gpu_available = [0.0] * self.num_gpus

        # YOUR CODE HERE: implement priority-based scheduling
        for req in results:
            gpu_idx = int(np.argmin(gpu_available))
            start = max(req['arrival_time'], gpu_available[gpu_idx])

            prefill_time = req['prompt_len'] / self.prefill_speed
            req['prefill_end'] = start + prefill_time
            req['ttft'] = req['prefill_end'] - req['arrival_time']

            # Apply thinking budget
            actual_thinking = min(req['thinking_tokens'], self.max_thinking)
            actual_total = actual_thinking + req['response_tokens']

            decode_time = actual_total * self.decode_tpot
            req['complete_time'] = req['prefill_end'] + decode_time
            req['total_latency'] = req['complete_time'] - req['arrival_time']
            req['thinking_used'] = actual_thinking
            req['preempted'] = actual_thinking < req['thinking_tokens']

            gpu_available[gpu_idx] = req['complete_time']

        return results


# Test your scheduler
scheduler = ReasoningScheduler(num_gpus=4, max_thinking_tokens=10000)
scheduler.train_predictor(workload[:500])
scheduled = scheduler.schedule(small_workload)

print("Custom Reasoning Scheduler Results:")
summarize_results(scheduled, "Custom Scheduler")

preempted = sum(1 for r in scheduled if r.get('preempted', False))
print(f"  Preempted requests: {preempted}/{len(scheduled)} ({preempted/len(scheduled):.1%})")

---
## Assignment 2: Implement Thinking Token Budget Control

In [ ]:
# ASSIGNMENT 2: Thinking Token Budget Control

class AdaptiveBudgetController:
    """
    TODO: Implement adaptive thinking budget that:
    1. Monitors thinking quality over time
    2. Adjusts budget based on task difficulty
    3. Implements early stopping when thinking is "converging"
    4. Respects cost and latency constraints
    """

    def __init__(self, cost_per_token: float = 0.00001,
                 max_cost_per_request: float = 0.50,
                 max_latency_s: float = 60.0):
        self.cost_per_token = cost_per_token
        self.max_cost = max_cost_per_request
        self.max_latency = max_latency_s
        self.history = []  # Track budget decisions and outcomes

    def compute_budget(self, task: str, prompt_len: int,
                        user_preference: str = 'balanced') -> dict:
        """
        TODO: Compute optimal thinking budget.

        Returns:
        - thinking_budget: max thinking tokens
        - early_stop_threshold: cosine similarity threshold for convergence
        - estimated_cost: dollar cost
        - estimated_latency: seconds
        """
        # YOUR CODE HERE
        task_bases = {
            'simple_qa': 300,
            'math_problem': 3000,
            'complex_reasoning': 10000,
            'code_generation': 5000,
            'creative_writing': 1500,
        }
        pref_mult = {'fast': 0.3, 'balanced': 1.0, 'thorough': 2.5}

        base = task_bases.get(task, 3000)
        budget = int(base * pref_mult.get(user_preference, 1.0))

        # Cost constraint
        max_by_cost = int(self.max_cost / self.cost_per_token)
        # Latency constraint (30ms per token)
        max_by_latency = int(self.max_latency / 0.03)

        budget = min(budget, max_by_cost, max_by_latency)

        return {
            'thinking_budget': budget,
            'early_stop_threshold': 0.95,
            'estimated_cost': budget * self.cost_per_token,
            'estimated_latency': budget * 0.03,
        }

    def simulate_early_stopping(self, total_thinking: int, budget: int) -> dict:
        """
        TODO: Simulate early stopping based on "convergence" detection.
        In practice, this would monitor the hidden state stability.
        """
        # Simulate convergence: quality plateaus after some fraction of tokens
        convergence_point = int(total_thinking * np.random.uniform(0.4, 0.9))
        stopped_at = min(budget, convergence_point)

        quality_if_full = 1.0
        quality_stopped = simulate_quality(total_thinking, stopped_at)

        return {
            'stopped_at': stopped_at,
            'quality': quality_stopped,
            'tokens_saved': max(0, total_thinking - stopped_at),
            'savings_pct': max(0, (total_thinking - stopped_at) / total_thinking),
        }


# Test adaptive budget controller
controller = AdaptiveBudgetController(cost_per_token=0.00001, max_cost_per_request=0.30)

print(f"Adaptive Budget Results:")
print(f"{'Task':>25} {'Pref':>10} {'Budget':>8} {'Cost':>8} {'Latency':>10}")
print("-" * 65)
for task in ['simple_qa', 'math_problem', 'complex_reasoning']:
    for pref in ['fast', 'balanced', 'thorough']:
        result = controller.compute_budget(task, 500, pref)
        print(f"{task:>25} {pref:>10} {result['thinking_budget']:>8} "
              f"${result['estimated_cost']:>6.3f} {result['estimated_latency']:>8.1f}s")

# Simulate early stopping savings
print(f"\nEarly Stopping Savings:")
total_savings = []
for req in workload[:200]:
    budget_info = controller.compute_budget(req['task'], req['prompt_len'])
    stop_result = controller.simulate_early_stopping(
        req['thinking_tokens'], budget_info['thinking_budget']
    )
    total_savings.append(stop_result)

avg_quality = np.mean([s['quality'] for s in total_savings])
avg_savings = np.mean([s['savings_pct'] for s in total_savings])
print(f"  Average quality preserved: {avg_quality:.2%}")
print(f"  Average tokens saved: {avg_savings:.1%}")

---
## Assignment 3: Benchmark Reasoning Model Serving

In [ ]:
# ASSIGNMENT 3: Reasoning Model Serving Benchmark

class ReasoningBenchmark:
    """
    TODO: Build a comprehensive benchmark for reasoning model serving.

    Metrics to track:
    - TTFT (Time to First Token)
    - TPOT (Time per Output Token)
    - Total latency distribution
    - Throughput at different SLO targets
    - Cost efficiency (tokens per dollar)
    - Quality vs latency trade-off
    """

    def __init__(self, workload: List[dict]):
        self.workload = workload

    def run_benchmark(self, server, name: str) -> dict:
        """
        TODO: Run benchmark and collect metrics.
        """
        # YOUR CODE HERE
        results = server.serve_fcfs(self.workload) if hasattr(server, 'serve_fcfs') else server.serve(self.workload)

        valid = [r for r in results if 'total_latency' in r]
        lats = [r['total_latency'] for r in valid]
        ttfts = [r['ttft'] for r in valid if 'ttft' in r]

        # SLO attainment at different targets
        slo_targets = [5, 10, 30, 60, 120]  # seconds
        slo_attainment = {}
        for target in slo_targets:
            met = sum(1 for l in lats if l <= target)
            slo_attainment[f'{target}s'] = met / len(lats)

        return {
            'name': name,
            'avg_latency': np.mean(lats),
            'p50_latency': np.median(lats),
            'p95_latency': np.percentile(lats, 95),
            'p99_latency': np.percentile(lats, 99),
            'avg_ttft': np.mean(ttfts) if ttfts else 0,
            'throughput': len(valid) / (max(r.get('complete_time', 0) for r in valid) -
                                        min(r['arrival_time'] for r in valid)),
            'slo_attainment': slo_attainment,
        }

    def compare(self, configs: List[Tuple]) -> None:
        """
        TODO: Compare multiple server configurations.

        configs: list of (server, name) tuples
        """
        all_results = []
        for server, name in configs:
            result = self.run_benchmark(server, name)
            all_results.append(result)

        print(f"\nBenchmark Comparison:")
        print(f"{'Config':>20} {'Avg Lat':>10} {'P95':>10} {'P99':>10} "
              f"{'Throughput':>10} {'TTFT':>10}")
        print("-" * 75)
        for r in all_results:
            print(f"{r['name']:>20} {r['avg_latency']:>8.1f}s {r['p95_latency']:>8.1f}s "
                  f"{r['p99_latency']:>8.1f}s {r['throughput']:>8.2f}/s "
                  f"{r['avg_ttft']*1000:>8.0f}ms")

        print(f"\nSLO Attainment:")
        print(f"{'Config':>20}", end='')
        for target in ['5s', '10s', '30s', '60s', '120s']:
            print(f"{target:>10}", end='')
        print()
        for r in all_results:
            print(f"{r['name']:>20}", end='')
            for target in ['5s', '10s', '30s', '60s', '120s']:
                print(f"{r['slo_attainment'].get(target, 0):>9.1%}", end='')
            print()


# Run benchmark
bench = ReasoningBenchmark(small_workload)

configs = [
    (ReasoningModelServer(num_gpus=4, decode_tpot_ms=30), '4 GPU, 30ms/tok'),
    (ReasoningModelServer(num_gpus=8, decode_tpot_ms=30), '8 GPU, 30ms/tok'),
    (ReasoningModelServer(num_gpus=4, decode_tpot_ms=15), '4 GPU, 15ms/tok'),
]

bench.compare(configs)

## Summary

### Key Takeaways

1. **Reasoning models** produce highly variable output lengths (heavy-tailed distribution), making traditional scheduling insufficient

2. **Output length prediction** enables better scheduling but is inherently uncertain -- robust strategies must handle prediction errors

3. **Thinking token budgets** enable cost/latency control with diminishing quality loss due to the logarithmic quality-budget relationship

4. **MLFQ-style scheduling** with aging and preemption helps balance fairness and throughput for mixed reasoning workloads

5. **Early stopping** can save 20-50% of thinking tokens with minimal quality impact

### Further Reading
- DeepSeek-R1: https://arxiv.org/abs/2401.02954
- Efficient Reasoning in LLMs: Scheduling and Budget Control